In [1]:
from datasets import load_dataset
from langchain_community.llms import Ollama
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings  # update ini
from tqdm import tqdm
import os

In [3]:
# Load dataset evaluasi
eval_dataset = load_dataset("json", data_files="/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/03_finetune_dataset/data/split/ragas_eval.jsonl")["train"]
eval_dataset[0]

{'question': 'Apa tujuan Manajemen Sumber Daya Manusia menurut Pasal 20?',
 'ground_truth': 'Berdasarkan Peraturan Bupati Nomor 45 Tahun 2022 tentang Sistem Pemerintahan Berbasis Elektronik Dalam Penyelenggaraan Pemerintahan Daerah, Pasal 20 menyatakan bahwa:\n\nPasal 20 Manajemen Sumber Daya Manusia sebagaimana dimaksud dalam Pasal 15 ayat (2) huruf e dilakukan untuk menjamin ketersediaan dan meningkatkan kompetensi pegawai ASN dalam penyelenggaraan SPBE melalui pendidikan dan pelatihan.\n\n(Sumber: Peraturan Bupati Nomor 45 Tahun 2022, Pasal 20)',
 'answer': 'Berdasarkan Peraturan Bupati Nomor 45 Tahun 2022 tentang Sistem Pemerintahan Berbasis Elektronik Dalam Penyelenggaraan Pemerintahan Daerah, Pasal 20 menyatakan bahwa:\n\nPasal 20 Manajemen Sumber Daya Manusia sebagaimana dimaksud dalam Pasal 15 ayat (2) huruf e dilakukan untuk menjamin ketersediaan dan meningkatkan kompetensi pegawai ASN dalam penyelenggaraan SPBE melalui pendidikan dan pelatihan.',
 'contexts': ['(Sumber:Peratu

In [5]:
# Load retriever (dari index yang baru dibuat)
embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large") 

In [6]:
vectorstore = FAISS.load_local("/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/04_rag_corpus/faiss_index_lexindo", embeddings, allow_dangerous_deserialization=True)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [7]:
# LLM via Ollama
llm_base = Ollama(model="llama3.2:1b-instruct-q4_K_M", temperature=0.3, num_ctx=8192)  # base
llm_lexindo = Ollama(model="hf.co/nxvay/lexindo_2e:q4_k_m", temperature=0.3, num_ctx=8192)  # finetuned

/var/folders/_w/v4htfwfd40ggzfmlgw22dk2m0000gn/T/ipykernel_49326/1437017268.py:2: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm_base = Ollama(model="llama3.2:1b-instruct-q4_K_M", temperature=0.3, num_ctx=8192)  # base


In [8]:
def generate_no_rag(question, model):
    prompt = f"""Pertanyaan: {question}
Jawaban akurat dan lengkap berdasarkan pengetahuan hukum daerah Indonesia:"""
    return model.invoke(prompt).strip()

def generate_with_rag(question, retriever, model):
    retrieved_docs = retriever.invoke(question)
    context_str = "\n\n".join([doc.page_content for doc in retrieved_docs])
    prompt = f"""Gunakan hanya informasi dari konteks berikut untuk jawab pertanyaan secara akurat. Cantumkan sumber jika relevan.
Konteks:
{context_str}

Pertanyaan: {question}
Jawaban:"""
    return model.invoke(prompt).strip()

In [9]:
# Generate (mulai kecil dulu, misal 200 samples)
subset = eval_dataset.select(range(200))  # atau full: eval_dataset

In [ ]:
# Base model no-RAG
subset = subset.map(lambda x: {"answer_base": generate_no_rag(x["question"], llm_base)}, batched=False)

# Finetuned no-RAG (kalau 'answer' lama sudah dari sini, skip atau overwrite)
subset = subset.map(lambda x: {"answer_finetuned_no_rag": generate_no_rag(x["question"], llm_lexindo)})

# Finetuned + RAG (fresh retrieve)
subset = subset.map(lambda x: {"answer_finetuned_rag": generate_with_rag(x["question"], retriever, llm_lexindo)})

# Update 'contexts' kalau ingin fresh (opsional)
subset = subset.map(lambda x: {"contexts": [doc.page_content for doc in retriever.invoke(x["question"])]})

# Simpan hasil
subset.to_json("/Users/novay/Applications/Project/Enterwind/lexindo/python/lexindo-llm/05_evaluation_set/ragas_eval_with_answers.jsonl")
print("Jawaban generated dan disimpan!")